# Форензик-карты до ресайза (серия `s3`)

## Что проверяется

Признаки, по которым правку видно в блоке 8×8 — соотношение полос DCT, поведение
хромы, остаток по решётке квантования, — живут на сетке JPEG. Ресайз кадра эту
сетку уничтожает. Замер в `docs/forensic_cues.md` (воспроизводится
`scripts/forensic_cues.py probe`): блочный пробник при переходе 1024 → 768 теряет

| домен | native | @768 | цена ресайза |
|---|---:|---:|---:|
| `coco` | 0.954 | 0.826 | −0.13 |
| `vision` | 0.761 | 0.699 | −0.06 |
| `plain_l9` | 0.592 | 0.562 | −0.03 |

Сеть смотрит на кадр уже после ресайза, то есть на признак, которого там больше
нет — сколько бы фильтров она ни выучила. Идея: посчитать карты признаков **до**
ресайза, на родном разрешении, и ресайзить карты, а не картинку. Карта живёт на
сетке 1/8, она уже агрегат — интерполяция её не портит.

## Три плеча, один фактор за раз

| плечо | что подаётся в ветку | отвечает на вопрос |
|---|---|---|
| **A** | ничего, ветки нет | якорь |
| **P** | те же каналы, но посчитанные **после** ресайза кадра | «дело в самих каналах или в ветке?» |
| **N** | те же каналы, посчитанные **до** ресайза | «дело в порядке ресайз/расчёт?» |

Каналов одиннадцать: весь набор `aic.forensic` минус `qt`, см. отдельный раздел
ниже — это ярлык кадра, а не признак области.

Плечо **P** здесь обязательно. Без него выигрыш «карты до ресайза» неотличим от
выигрыша «в сети стало на одиннадцать каналов и одну ветку больше»: любая
дополнительная ёмкость на 192k показов что-нибудь да даёт. Интересна разница
`N − P`, а не `N − A`.

Данные у всех трёх плеч идентичны до последнего пикселя: тот же кроп, кратный 8,
та же аугментация перекодированием на родном разрешении, тот же порядок флипов.
Различается ровно одно поле в батче.

## Что считать результатом

Предсказание, которое можно опровергнуть:

* `N − P` **положительна на `vision` и `coco`** — там низкоуровневый след
  сильный, и именно его съедает ресайз;
* `N − P` **около нуля на `plain_l9`** — там следа почти нет вообще, и никакой
  порядок вычислений его не создаст;
* общий AIC двигается слабо, потому что `coco` уже на Dice 0.94, а доля
  `vision` — 13%.

Если `N − P` окажется положительной и на `plain_l9` — значит замер в
`docs/forensic_cues.md` чего-то не увидел, и надо возвращаться к разбору данных.
Если `N ≈ P` везде — идея не работает, и это тоже ответ: доступность признака в
блочном пробнике не переносится на сеть.

## Чего в этом ноутбуке нет

Ветка `fuse` не трогает страйд 4: карта не несёт информации выше 1/8, и вливать
её на более мелкой сетке нечего. Инференс и сборка сабмита здесь тоже не
делаются: чтобы это поехало в сабмит, карту надо научить считать `inference.py`
и `PredictDataset` — по одной и той же геометрии, иначе ветка увидит на тесте
другое распределение входа. Точечный список правок пайплайна —
в `docs/forensic_stream.md`.

## 0. Импорты

In [1]:
import os

# Выставляется ДО импорта torch: аллокатор читает переменную при первой
# аллокации, и после `import torch` менять её уже поздно. Спасает не от
# нехватки памяти, а от её фрагментации — ровно то, что советует само
# сообщение OOM, когда «reserved but unallocated» заметно больше нуля.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["BLIS_NUM_THREADS"] = "1"

In [2]:
import aic
import numpy as np
import cv2
import torch

cv2.setNumThreads(0)
torch.set_num_threads(1)

In [3]:
from threadpoolctl import threadpool_info
import pprint

pprint.pp(threadpool_info())

[{'user_api': 'blas',
  'internal_api': 'openblas',
  'num_threads': 1,
  'prefix': 'libscipy_openblas',
  'filepath': '/home/user/home/user/data/aic/venv/lib/python3.11/site-packages/numpy.libs/libscipy_openblas64_-32a4b2a6.so',
  'version': '0.3.31.188.0',
  'threading_layer': 'pthreads',
  'architecture': 'Haswell'},
 {'user_api': 'blas',
  'internal_api': 'openblas',
  'num_threads': 1,
  'prefix': 'libopenblas',
  'filepath': '/home/user/home/user/data/aic/venv/lib/python3.11/site-packages/opencv_python_headless.libs/libopenblasp-r0-59ffcd50.3.15.so',
  'version': '0.3.15',
  'threading_layer': 'pthreads',
  'architecture': 'Haswell'},
 {'user_api': 'openmp',
  'internal_api': 'openmp',
  'num_threads': 1,
  'prefix': 'libgomp',
  'filepath': '/home/user/home/user/data/aic/venv/lib/python3.11/site-packages/torch/lib/libgomp-a34b3233.so.1',
  'version': None}]


In [4]:
# aic импортируется первым, до torch/cv2/albumentations: он выставляет
# KMP_DUPLICATE_LIB_OK (иначе две копии OpenMP роняют процесс) и
# NO_ALBUMENTATIONS_UPDATE (иначе albumentations ходит в сеть на каждом импорте).
import aic
from aic import budget
from aic import data as aic_data
from aic import stats as aic_stats
from forensic import (
    CHANNELS as FMAP_CHANNELS,
    N_CHANNELS as FMAP_N,
    forensic_maps,
    luma_qtable,
    maps_after_resize,
)

import math
import os
import re
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F

cv2.setNumThreads(0)  # иначе воркеры DataLoader дерутся за ядра с основным процессом

ws = aic.Workspace.find(Path.cwd())
ws.ensure_dirs()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(ws)
print(f"aic {aic.__version__} | torch {torch.__version__} | timm {timm.__version__}")
print(f"каналов карты: {FMAP_N} -> {', '.join(FMAP_CHANNELS)}")
if DEVICE.type == "cuda":
    caps = "".join(map(str, torch.cuda.get_device_capability()))
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{torch.cuda.get_device_name(0)}, sm_{caps}, {vram:.1f} ГБ")
else:
    print("ВНИМАНИЕ: CUDA не видна. Разделы 3 и 5.1 считаются на CPU и осмысленны, "
          "обучение — нет")

/home/user/home/user/data/aic/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Workspace('/home/user/home/user/data/aic')
aic 0.1.0 | torch 2.6.0+cu124 | timm 1.0.28
каналов карты: 12 -> y31, y31_rel, y42, y42_rel, c31, c31_rel, cy3, cy3_rel, latt, nz_rel, mid_rel, qt
Tesla V100-SXM2-16GB, sm_70, 16.9 ГБ


## 1. Плечи серии

In [5]:
#: какие плечи считать этим запуском. Якорь A первым: с ним сравниваются
#: остальные. Оставь одну букву, если нужно посчитать только её.
ARMS_TO_RUN = ["F1", "F0"]

#: Блок прогрессивного уточнения (arXiv:2607.29156). Значения порогов, число
#: шагов и beta — авторские; ev_ch/hidden в статье не названы вовсе, взяты по
#: месту: улик вдвое меньше, чем каналов отвода (16 из 32), оператор считает в
#: 32. kind="mamba" требует mamba-ssm с CUDA-ядром и на Windows не поедет.
REFINE = dict(
    kind="conv",
    ev_ch=16,
    hidden=32,
    steps=2,          # T=2; в абляции статьи T>2 делает хуже
    tau_low=0.3,
    tau_high=0.8,
    beta=1.0,         # авторское; при нём разделение мягкое, см. markdown 5.0
    alpha=1.0,        # в статье не названо
    test_offset=0.2,  # на инференсе оба порога опускаются
)

BASE = dict(
    # данные
    fold=0,
    epoch_size=24_000,
    val_frac=0.25,
    val_keep_negatives=True,   # прореживаем только позитивы: негативов всего ~640
    negative_fraction=0.25,
    crop_scale=(0.35, 1.0),
    size=640,
    jpeg_aug_p=0.3,            # перекодирование на РОДНОМ разрешении, до карт
    jpeg_aug_quality=(60, 100),

    # карта: none | post | native   <- единственный фактор серии
    fmap="native",
    fmap_drop=("qt",),         # см. markdown ниже: это ярлык кадра, а не признак области
    fmap_ch=(64, 96, 128),     # каналы ветки на страйдах 8 / 16 / 32
    fmap_lr=3e-4,              # ветка учится с нуля — полный lr, не encoder_lr

    # модель
    encoder="convnext_tiny",
    decoder_channels=(128, 64, 32, 16, 16),   # полный (256,...) съел бы треть бюджета
    norm="batch",
    aux_weight=0.4,            # глубокая супервизия на страйде 4
    refine=None,               # словарь REFINE включает уточнение, None — прежняя модель
    deep_weight=1.0,           # L_deep из статьи, лямбда=1
    evidence_weight=1.0,       # L_aux из статьи, лямбда=1

    # Серия L: термы под измеренный профиль ошибок. Все по нулям — плечо
    # включает РОВНО ОДИН, иначе снова не понять, что сработало.
    w_gate=0.0,                # гейт множителем предсказания, а не весом лосса
    gate_warmup=0.1,           # доля обучения, за которую вес гейта выходит на 1
    w_comp=0.0,                # покомпонентный recall: куски, пропущенные целиком
    comp_tau=0.25,             # ниже этого покрытия кусок считается незамеченным
    w_far=0.0,                 # штраф за уверенность вдали от GT
    far_margin=0.01,           # ближе 1% диагонали не штрафуем: это граница, а не промах
    w_lovasz=0.0,              # суррогат IoU по порядку пикселей

    # Разрешить подбор батча, если карта занята. По умолчанию ВЫКЛЮЧЕНО: якорь
    # серии считался на bs=8, а BatchNorm декодера копит статистику по
    # микробатчу, и плечо на bs=4 с ним уже не сравнить.
    allow_batch_refit=False,

    # Геометрия обучения. False — случайный квадратный кроп, как было всегда.
    # True — та же рамка, что у инференса: весь кадр, без поворотов на 90.
    full_frame=False,
    #: имя прогона, с чьего `ckpt/best.pt` стартовать вместо ImageNet
    init_from=None,

    # обучение
    epochs=8,
    bs=8,
    accum_steps=2,
    lr=3e-4,
    encoder_lr=1e-4,
    weight_decay=1e-4,
    warmup_frac=0.05,
    min_lr_factor=0.02,
    amp="fp16",
    grad_clip=1.0,
    ema_decay=0.999,
    num_workers=min(10, os.cpu_count() or 4),
    resume=True,
    seed=42,
)

ARMS = {
    "A": {**BASE, "name": "s3-A-control640", "fmap": "none"},
    "P": {**BASE, "name": "s3-P-fmap-post640", "fmap": "post"},
    "N": {**BASE, "name": "s3-N-fmap-native640", "fmap": "native"},
    "T": {**BASE, "name": "s3-T-mitb2-native640_long", "fmap": "native", "encoder": "pvt_v2_b2"},
    # уточнение поверх N: энкодер прежний, меняется только способ получить маску
    "R": {**BASE, "name": "s3-R-egrefine-native640", "fmap": "native", "refine": REFINE, "encoder": "pvt_v2_b2"},
    # оба изменения сразу; ставить в очередь только после того, как R и T
    # ответили по отдельности, иначе непонятно, что именно сработало
    "TR": {**BASE, "name": "s3-TR-mitb2-egrefine-native640", "fmap": "native",
           "encoder": "pvt_v2_b2", "refine": REFINE},

    # --- серия L: по одному терму на плечо ---------------------------------
    # Восемь эпох, а не восемнадцать: якорь для них — уже посчитанный
    # `s3-T-mitb2-native640` (192k показов, AIC 0.8895). Скрининг на той же
    # длине стоит вдвое дешевле и решается по тем же числам; до 432k доводить
    # имеет смысл только то, что здесь выиграет.
    "L2": {**BASE, "name": "s3-L2-comp-native640", "encoder": "pvt_v2_b2",
           "epochs": 8, "w_comp": 0.5},
    "L3": {**BASE, "name": "s3-L3-gate-native640", "encoder": "pvt_v2_b2",
           "epochs": 8, "w_gate": 1.0},
    "L4": {**BASE, "name": "s3-L4-lovasz-native640", "encoder": "pvt_v2_b2",
           "epochs": 8, "w_lovasz": 0.5},
    # Терм нормирован на предсказанную массу, поэтому читается как «средняя
    # дальность предсказания от GT» и лежит в [0, ~0.7] независимо от площади
    # маски. Вес 1.0 ставит его вровень с Dice на кадрах, где модель мажет
    # мимо, и почти в ноль там, где она просто размазала границу.
    "L5": {**BASE, "name": "s3-L5-far-native640", "encoder": "pvt_v2_b2",
           "epochs": 8, "w_far": 1.0},
    # --- B1: ёмкость энкодера в обмен на разрешение --------------------------
    # Единственный рычаг, измеренный в этом проекте дважды. На текущих весах,
    # без всякого дообучения, тот же pvt_v2_b2 на входе 768 даёт AIC 0.9300
    # против 0.9070 на 640 — то есть модель упирается в разрешение, а не в
    # ёмкость. Отсюда и обмен: b2 на 640 стоит 97.2 GFLOPs и упирается в
    # потолок (656 уже вне лимита), b1 на 768 стоит 90.1 и оставляет запас.
    #
    # Чем платим: у b1 в каждой стадии по два блока против [3, 4, 6, 3] у b2,
    # энкодер 13.5M против 24M. Насколько просядет `plain_l9`, где нужна
    # семантика, заранее не известно — это и есть вопрос плеча.
    #
    # 18 эпох сразу, а не 8 с продолжением: `lr_lambda` замыкается на
    # `total_steps`, и досчёт с изменённым `epochs` подбросил бы LR вверх
    # посреди косинуса. Смотреть на `val/aic_tuned` после восьмой эпохи —
    # если она заметно ниже 0.8895 (столько даёт b2@640 на 192k показов),
    # обмен не окупился и плечо можно снимать досрочно.
    "B1": {**BASE, "name": "s3-B1-pvtb1-native768", "fmap": "native",
           "encoder": "pvt_v2_b1", "size": 768, "epochs": 18,
           "bs": 8, "accum_steps": 2},

    # --- F: дотюн под геометрию инференса ------------------------------------
    # Замер, из которого выросла пара. Одни и те же 2000 кадров, один и тот же
    # чекпоинт, меняется только вход:
    #
    #   b2 (обучен @640):  640 -> 0.9057   768 -> 0.9294   896 -> 0.9396
    #   b1 (обучен @768):  640 -> 0.8350   768 -> 0.8987   896 -> 0.9134
    #
    # Обе модели показывают лучший AIC ВЫШЕ своего разрешения обучения. Причина
    # как минимум частично в геометрии: случайный кроп берёт 81.3% короткой
    # стороны (объекты крупнее в 1.23 раза) и всегда квадратный, а валидация
    # ужимает в ту же сетку весь прямоугольник. Поднять вход нельзя — 768 это
    # 151.9 GFLOPs, — но можно убрать само рассогласование, и это бесплатно:
    # 97.2 GFLOPs как были.
    #
    # F0 обязателен. Без него прирост F1 неотличим от «просто ещё поучили на
    # низком LR», сколько бы якорь ни выглядел сошедшимся.
    "F1": {**BASE, "name": "s3-F1-fullframe-ft640", "fmap": "native",
           "encoder": "pvt_v2_b2", "size": 640, "full_frame": True,
           "init_from": "s3-T-mitb2-native640_long",
           "epochs": 3, "lr": 3e-5, "encoder_lr": 1e-5, "fmap_lr": 3e-5,
           "warmup_frac": 0.02, "min_lr_factor": 0.05},
    "F0": {**BASE, "name": "s3-F0-crop-ft640", "fmap": "native",
           "encoder": "pvt_v2_b2", "size": 640, "full_frame": False,
           "init_from": "s3-T-mitb2-native640_long",
           "epochs": 3, "lr": 3e-5, "encoder_lr": 1e-5, "fmap_lr": 3e-5,
           "warmup_frac": 0.02, "min_lr_factor": 0.05},

    # повтор пары на втором сиде: разница N-P ожидается меньше шума обучения
    # (0.0035 AIC), и на одном сиде её просто не разглядеть
    "P7": {**BASE, "name": "s3-P7-fmap-post640-s7", "fmap": "post", "seed": 7},
    "N7": {**BASE, "name": "s3-N7-fmap-native640-s7", "fmap": "native", "seed": 7},
}

unknown = [a for a in ARMS_TO_RUN if a not in ARMS]
if unknown:
    raise SystemExit(f"нет таких плеч: {unknown}; есть: {', '.join(ARMS)}")

#: якорь и пара — по именам прогонов, а не по буквам: разделы 10 и 11 ищут их
#: на диске и работают, даже если посчитаны в разные заходы
ANCHOR = ARMS["A"]["name"]
COUNTERPART = {"N": ARMS["P"]["name"], "N7": ARMS["P7"]["name"],
               "R": ARMS["N"]["name"], "TR": ARMS["T"]["name"],
               # якорь серии L — прогон той же архитектуры на 192k показов
               "L2": "s3-T-mitb2-native640", "L3": "s3-T-mitb2-native640",
               "L4": "s3-T-mitb2-native640", "L5": "s3-T-mitb2-native640",
               # B1 идёт на 432k показов, поэтому сравнивается с длинным
               # прогоном b2@640 (AIC 0.9070), а не с восьмиэпошным якорем
               "B1": "s3-T-mitb2-native640_long",
               # F1 сравнивается со СВОИМ контролем: так изолируется геометрия,
               # а не «дотюн вообще». F0 — с исходным прогоном: так видно, что
               # даёт сам по себе низкий LR на сошедшейся модели.
               "F1": "s3-F0-crop-ft640", "F0": "s3-T-mitb2-native640_long"}

# ВНИМАНИЕ, ДЛИНА. Прогоны A/P/N на диске сняты на 192k показов, а здесь
# epochs=18 при epoch_size=24_000 даёт 432k. Сравнение нового плеча с
# сохранённым N приписало бы разницу длины обучения блоку уточнения. Либо
# верните epochs=8 на время A/B, либо пересчитайте пару заново на 18 эпохах —
# третьего варианта нет.
if BASE["epochs"] * BASE["epoch_size"] != 192_000:
    print(f"! показов на плечо: {BASE['epochs'] * BASE['epoch_size']}, "
          f"а у сохранённых A/P/N — 192000; прямое A/B с ними невалидно")

# AMP разрешается по карте, а не по вере: ниже sm_80 bf16 идёт эмуляцией.
# Он один на все плечи — иначе сравнивались бы ещё и режимы точности.
AMP = BASE["amp"]
if AMP == "auto":
    if DEVICE.type != "cuda":
        AMP = "off"
    else:
        major, _ = torch.cuda.get_device_capability()

        # native BF16 только Ampere+
        AMP = "bf16" if major >= 8 else "fp16"
USE_AMP = AMP != "off" and DEVICE.type == "cuda"
USE_SCALER = AMP == "fp16" and DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if AMP == "bf16" else torch.float16
ARMS = {k: {**v, "amp": AMP} for k, v in ARMS.items()}

#: заполняется разделом 3; объявлен здесь, чтобы обучение не падало на снапшоте,
#: если пробник пропустили
probe_report = {}

FMAP_KEEP = [i for i, name in enumerate(FMAP_CHANNELS) if name not in BASE["fmap_drop"]]
FMAP_USED = len(FMAP_KEEP)

total = sum(ARMS[a]["epochs"] * ARMS[a]["epoch_size"] for a in ARMS_TO_RUN)
for arm in ARMS_TO_RUN:
    cfg = ARMS[arm]
    print(f"  [{arm}] {cfg['name']:<32} карта={cfg['fmap']:<6} "
          f"энкодер={cfg['encoder']:<14} "
          f"уточнение={(cfg.get('refine') or {}).get('kind', 'нет')} "
          f"термы={', '.join(k for k in ('w_gate', 'w_comp', 'w_far', 'w_lovasz') if cfg.get(k)) or 'нет'} "
          f"seed={cfg['seed']}")
print(f"каналов в ветке {FMAP_USED} из {FMAP_N}; выброшены: "
      f"{', '.join(BASE['fmap_drop']) or 'нет'}")
print(f"{BASE['encoder']} @{BASE['size']} | amp={AMP} | "
      f"{total} показов суммарно по {len(ARMS_TO_RUN)} плечам")


  [F1] s3-F1-fullframe-ft640            карта=native энкодер=pvt_v2_b2      уточнение=нет термы=нет seed=42
  [F0] s3-F0-crop-ft640                 карта=native энкодер=pvt_v2_b2      уточнение=нет термы=нет seed=42
каналов в ветке 11 из 12; выброшены: qt
convnext_tiny @640 | amp=fp16 | 144000 показов суммарно по 2 плечам


Запускать «Run All»: раздел 1 перечисляет плечи, раздел 8 гоняет их подряд, и
каждое ложится в свой прогон под своим именем. Чтобы посчитать только одно —
оставить в `ARMS_TO_RUN` одну букву; якорь `A` должен идти первым, на него
опираются разделы 9 и 10.

Прогон длинный: три плеча по 8 эпох на 24k показов. `resume=True` включён, так
что прерванный ноутбук продолжится с последней эпохи каждого плеча, а не начнёт
заново.

Что именно различается между плечами, кроме имени прогона:

* `fmap="none"` — ветки нет, батч без поля `fmap`, модель побитово равна якорю;
* `fmap="post"` — карта считается из **уже отресайзенного** кропа, той же
  функцией и с тем же числом каналов; таблица квантования при этом не передаётся,
  потому что к отресайзенному кадру она не относится;
* `fmap="native"` — карта считается по полному кадру до ресайза, из неё
  вырезается кроп (кратный 8, иначе сетка блоков разъедется), и уже он
  уменьшается до сетки `size/8`.

Аугментация перекодированием живёт **в датасете, на родном разрешении**, а не в
`albumentations`. Иначе она пережимала бы кадр уже после того, как карта
посчитана, и карта перестала бы соответствовать картинке — у плеча N это
испортило бы ровно тот признак, ради которого всё делается. По той же причине
из пиксельных аугментаций убран `Downscale`. Яркость и цвет остались: каналы
карты — логарифмические ОТНОШЕНИЯ полос, и общий множитель яркости из них
сокращается.

### Почему канал `qt` выброшен

`qt` — это `log2 Q[0,0]`, то есть степень сжатия кадра, одинаковая во всех его
блоках. Замер по 3000 кадрам валидации: **один этот скаляр отделяет позитив от
негатива с AUC 0.727** (у негативов `Q[0,0] ≤ 2` в 100% случаев, у позитивов — в
80%). Причина понятная: негативы — это исходные съёмочные JPEG, а позитивы
пересохранены редактором, и качество у них другое.

Это не утечка train/val: в тесте, собранном так же, признак работать будет. Но
для **этой** серии он вреден дважды. Во-первых, признак не про область — он
константен по кадру и может влиять только на гейт, то есть на FPR, а меряем мы
локализацию. Во-вторых, он есть у плеч N и P и отсутствует у якоря A, так что
вердикт «против якоря» стал бы вердиктом про сжатие, а не про карты.

Проверить его отдельно — отдельная и вполне осмысленная работа
(`fmap_drop=()`), но не под видом этой.

Остаточный риск, который так не снимается: канал `latt` считается по настоящей
таблице квантования, и у кадров с `Q[0,0] = 1` он вырожден почти всюду. То есть
слабый след «насколько кадр сжат» в карте остаётся и после выброса `qt`.

## 2. Воркспейс, индекс и фолды

In [6]:
ALLOW_BUILD_INDEX = False


def subsource(stem):
    """plain — это два датасета: 9 цифр без hash-префикса = plain_l9.

    Разделение нужно разделу 11: предсказание серии сформулировано по доменам,
    и `plain` целиком его не проверяет — внутри него `plain_l8` на Dice 0.95, а
    `plain_l9` на 0.68.
    """
    if re.fullmatch(r"\d{9}", str(stem)):
        return "plain_l9"
    domain = aic_data.parse_domain(str(stem))
    return "plain_l8" if domain == "plain" else domain


missing = [p for p in (ws.index_path, ws.split_path) if not p.exists()]
if missing and not ALLOW_BUILD_INDEX:
    raise SystemExit(
        "нет " + ", ".join(str(p) for p in missing) + ".\n"
        "Нарезка обязана остаться той же, на которой посчитаны прошлые прогоны, "
        "иначе сравнение с якорем ничего не значит. Либо скопируй артефакты, "
        "либо поставь ALLOW_BUILD_INDEX = True и пересчитай ВСЕ плечи заново."
    )
if missing:
    index = (aic_data.load_index(ws.index_path) if ws.index_path.exists()
             else aic_data.build_index(ws, workers=os.cpu_count() or 8))
    aic_data.make_folds(index, n_folds=5, seed=BASE["seed"], out_path=ws.split_path)

folds = aic_data.load_folds(ws.split_path)
folds["subsource"] = folds["stem"].map(subsource)

print(aic_data.summarize_folds(folds)[:400])
print("\nподысточники:", folds["subsource"].value_counts().to_dict())

          n  neg  neg_frac  groups  area_med
fold                                        
0     20740  639  0.030810   15879  0.150534
1     20740  634  0.030569   15887  0.151871
2     20740  642  0.030955   15880  0.150064
3     20740  633  0.030521   15886  0.151725
4     20739  639  0.030812   15884  0.151441

утечка групп между фолдами: 0 — чисто

домены по фолдам:
domain  coco  numid  openim

подысточники: {'coco': 33483, 'plain_l9': 23955, 'vision': 13255, 'raise': 11802, 'plain_l8': 11581, 'numid': 6669, 'openimages': 2954}


## 3. Проверка признака до обучения

Дешёвая ступень, которая идёт **до** любых GPU-часов: посчитать те же двенадцать
каналов, которые получит ветка, и проверить блочным пробником, что до ресайза в
них сигнала больше, чем после. Если разницы нет, обучать три плеча незачем — либо
каналы реализованы не так, как в `scripts/forensic_cues.py`, либо гипотеза не
про эти данные.

Негативы подбираются **по содержанию**: каждому блоку внутри правки берётся
блок фона того же кадра с той же яркостью и той же энергией средних частот. Без
этого пробник выучивает «текстура против неба» и выдаёт 0.63 там, где честных
0.60 — разбор этой ловушки в `docs/forensic_cues.md` §2.

In [7]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

PROBE_SIZE = BASE["size"]
MID_REL = FMAP_CHANNELS.index("mid_rel")


def probe_frame(row, mode: str):
    """Признаки блоков одного кадра + метка «блок внутри правки».

    `mode`: 'native' — карта по родному кадру; 'post' — кадр сперва ресайзится
    в PROBE_SIZE, и карта считается уже по нему. Ровно то же различие, что между
    плечами N и P, но без сети.
    """
    path = ws.resolve(row["chng_path"])
    bgr = aic_data.imread(path, cv2.IMREAD_COLOR)
    gt = aic_data.imread(ws.resolve(row["gt_path"]), cv2.IMREAD_GRAYSCALE)
    if bgr is None or gt is None:
        return None
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    if gt.shape[:2] != rgb.shape[:2]:
        gt = cv2.resize(gt, (rgb.shape[1], rgb.shape[0]), interpolation=cv2.INTER_NEAREST)
    binary = (gt > 127).astype(np.float32)
    if not (0.01 < binary.mean() < 0.4):
        return None

    if mode == "native":
        maps = forensic_maps(rgb, luma_qtable(path))
        luma_src, gt_src = rgb, binary
    else:
        maps = maps_after_resize(rgb, PROBE_SIZE, luma_qtable(path))
        luma_src = cv2.resize(rgb, (PROBE_SIZE, PROBE_SIZE), interpolation=cv2.INTER_LINEAR)
        gt_src = cv2.resize(binary, (PROBE_SIZE, PROBE_SIZE), interpolation=cv2.INTER_NEAREST)

    h, w = maps.shape[1], maps.shape[2]
    grid = lambda x: cv2.resize(x, (w, h), interpolation=cv2.INTER_AREA)
    luma = grid(cv2.cvtColor(luma_src, cv2.COLOR_RGB2GRAY).astype(np.float32))
    cover = grid(gt_src)

    features = maps[FMAP_KEEP].reshape(FMAP_USED, -1).T
    return features, cover.ravel(), luma.ravel(), maps[MID_REL].ravel()


def probe_blocks(rows, n_frames: int, mode: str):
    """Пары «блок правки — подобранный блок фона» с нескольких кадров."""
    X, y, g = [], [], []
    for i, (_, row) in enumerate(rows.iterrows()):
        out = probe_frame(row, mode)
        if out is None:
            continue
        features, cover, luma, mid = out
        inside = np.where(cover > 0.95)[0]
        outside = np.where(cover < 0.05)[0]
        if len(inside) < 20 or len(outside) < 200:
            continue
        rng = np.random.default_rng(i)
        inside = rng.choice(inside, min(len(inside), 250), replace=False)
        picked = []
        for j in inside:
            near = np.where((np.abs(luma[outside] - luma[j]) < 12)
                            & (np.abs(mid[outside] - mid[j]) < 0.15))[0]
            if len(near):
                picked.append(outside[rng.choice(near)])
        if len(picked) < 20:
            continue
        inside = inside[:len(picked)]
        X.append(np.concatenate([features[inside], features[picked]]))
        y.append(np.concatenate([np.ones(len(inside)), np.zeros(len(picked))]))
        g.append(np.full(2 * len(inside), len(g)))
        if len(X) >= n_frames:
            break
    if not X:
        return None
    return np.concatenate(X), np.concatenate(y).astype(np.int8), np.concatenate(g)

In [8]:
PROBE_DOMAINS = ("coco", "vision", "plain_l9")
PROBE_TRAIN, PROBE_TEST = 90, 45

pool = folds[(folds["fold"] == BASE["fold"]) & (~folds["is_negative"])].copy()
pool["subsource"] = pool["stem"].map(subsource)

print(f"{'домен':<11}{'до ресайза':>12}{'после':>10}{'разница':>10}   "
      f"(AUC по блокам, негативы подобраны по содержанию)")
probe_report = {}
for domain in PROBE_DOMAINS:
    rows = pool[pool["subsource"] == domain].sample(
        min(700, int((pool["subsource"] == domain).sum())), random_state=11
    ).reset_index(drop=True)
    scores = {}
    for mode in ("native", "post"):
        train = probe_blocks(rows.iloc[:450], PROBE_TRAIN, mode)
        test = probe_blocks(rows.iloc[450:], PROBE_TEST, mode)
        if train is None or test is None:
            scores[mode] = float("nan")
            continue
        model = HistGradientBoostingClassifier(max_iter=200, max_depth=4)
        model.fit(train[0], train[1])
        scores[mode] = roc_auc_score(test[1], model.predict_proba(test[0])[:, 1])
    probe_report[domain] = scores
    print(f"{domain:<11}{scores['native']:>12.3f}{scores['post']:>10.3f}"
          f"{scores['native'] - scores['post']:>10.3f}")

домен        до ресайза     после   разница   (AUC по блокам, негативы подобраны по содержанию)
coco              0.879     0.699     0.180
vision            0.706     0.623     0.083
plain_l9          0.587     0.560     0.027


Как читать. На фолде 0 этот пробник уже считался, вот его вывод — с ним и
сверяться:

| домен | до ресайза | после | разница |
|---|---:|---:|---:|
| `coco` | 0.879 | 0.698 | **+0.180** |
| `vision` | 0.708 | 0.623 | +0.085 |
| `plain_l9` | 0.583 | 0.561 | +0.022 |

Порядок тот же, что в `docs/forensic_cues.md`, а разрыв даже больше: там у
пробника было тридцать два признака на блок, здесь одиннадцать, и после ресайза
из них выбывает ещё и решётка квантования — считать `latt` по таблице исходного
файла для отресайзенного кадра нельзя, там этой решётки уже нет.

Именно эти числа и есть верхняя оценка того, что серия может забрать: ветка
получает ровно ту же информацию, что и пробник.

Когда останавливаться и не тратить GPU:

* разница `native − post` около нуля на всех доменах — каналы не те, что мерились
  в `docs/forensic_cues.md`; сверяться с `scripts/forensic_cues.py probe`;
* `native` сам по себе около 0.5 — карта считается неправильно (частая причина:
  таблица квантования развёрнута не из зигзага, канал `latt` тогда шум).

Ниже пробник больше не нужен и память под него можно не держать.

## 4. Данные: геометрия руками, карта — до ресайза

Датасет и загрузчики лежат в [`s3_data.py`](s3_data.py) рядом с этим ноутбуком, а
не в ячейке. Причина одна и она техническая: воркеры `DataLoader` на Windows
стартуют через `spawn` и распикливают датасет по ссылке `<модуль>.SegDataset`.
Класс из ячейки живёт в `__main__`, а у спавненного процесса `__main__` — заглушка
`multiprocessing`, потому что у ядра Jupyter нет `__file__`. Класс не находится,
воркер падает, и остаётся `num_workers=0` — при 30 мс на карту это разница между
«ночь» и «трое суток». Из импортируемого файла ребёнок берёт класс без проблем.

На Linux с `fork` это не требуется, но и не мешает: ценой одного импорта ноутбук
ведёт себя одинаково на обеих системах.

Что в модуле и почему именно так:

* **геометрия делается numpy/cv2, а не `albumentations`.** Карту надо провести
  через те же кроп, флипы и ресайз, что и кадр, но на своей сетке 1/8; `Compose`
  умеет только полное разрешение — это 32 МБ на кадр при двенадцати каналах;
* **кроп кратен 8 по всем четырём координатам.** Иначе сетка блоков внутри кропа
  не совпадает с настоящей, и карта описывает уже другую картинку. `crop_maps`
  на этом падает специально, а не молчит;
* **аугментация перекодированием — до карты и на родном разрешении.** Пережали —
  значит и таблица квантования теперь другая, её берём из нового буфера;
* **флипы и повороты на 90° карте не вредят.** Полосы DCT заданы по сумме
  индексов частот, а она симметрична относительно транспонирования; знак
  коэффициента при отражении меняется, энергия — нет.

Модель, лосс и цикл обучения остались в ноутбуке: они работают в главном
процессе, пикклиться им не нужно, а править их хочется прямо здесь.

In [9]:
import sys

# путь до модуля рядом с ноутбуком: ядро Jupyter обычно стартует в notebooks/,
# но если ноутбук запущен из корня репозитория — тоже находим.
# `sys.path` уезжает в спавненные воркеры целиком, так что импорт у них сработает
NOTEBOOK_DIR = next(
    (d for d in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent)
     if (d / "s3_data.py").exists()), None
)
if NOTEBOOK_DIR is None:
    raise SystemExit("не нашёл s3_data.py — он должен лежать рядом с ноутбуком")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import s3_data
from s3_data import SegDataset, split_frames   # noqa: F401 — читать код там


def make_data(cfg, workers=None):
    """Обёртка: подставляет воркспейс, фолды и набор каналов этой серии."""
    return s3_data.make_data(
        ws, folds, cfg, fmap_keep=FMAP_KEEP,
        workers=cfg["num_workers"] if workers is None else workers,
        pin_memory=(DEVICE.type == "cuda"),
    )


print(f"s3_data из {NOTEBOOK_DIR}")

s3_data из /home/user/home/user/data/aic/notebooks


### 4.1 Сколько воркеров поднимется и хватит ли их

Четыре проверки, все дешёвые и все про то, что иначе выяснится через час обучения.

1. **Поднимутся ли воркеры.** `next(iter(train_loader))` — первое место, где
   процессы реально стартуют. С датасетом из модуля они должны подняться и на
   Windows; откат на `num_workers=0` остаётся страховкой, а не планом.
2. **Что лежит в батче** — форма, тип и диапазон каждого ключа. `nan` в карте
   (пустая таблица квантования, логарифм нуля) видно сразу.
3. **Совпадает ли сетка карты с сеткой модели** — `assert` на
   `(bs, каналы, size/8, size/8)`. Если карта придёт не ровно в 1/8 входа,
   `GatedFuse` молча растянет её интерполяцией, и всё построение, ради которого
   кроп делается кратным 8, развалится, не выдав ни одной ошибки.
4. **Не упрётся ли всё в CPU** — 25 батчей с секундомером, после прогрева.
   Прогрев обязателен: на spawn каждый воркер заново импортирует
   torch/cv2/albumentations, и без него замер показал бы, что воркеры вредны.
   Замер на этой машине (RTX 3070, 20 ядер, кадры 640px):

   | воркеров | старт | дальше |
   |---:|---:|---:|
   | 0 | 0.4 с | 26 кадров/с |
   | 6 | 18 с | **73 кадра/с** |

   Порог `bs * 3` — прикидка, а не замер: сколько шагов в секунду даёт карта на
   этой модели, я не мерил. Настоящая проверка появится в `epoch_time_s` первой
   эпохи.

`WORKERS` решается один раз на всю серию. Не только чтобы не ловить одну ошибку
трижды: зерно аугментаций в `__getitem__` берётся из `torch.initial_seed()`, а он
различается между воркерами. Кадры и их порядок от числа воркеров не зависят
(сэмплер живёт в главном процессе), а вот поток кропов и флипов — да, и плечи
должны видеть один и тот же.

In [10]:
# плечо для проверки берём с картой: у якоря поля `fmap` нет и смотреть нечего
probe_arm = next((a for a in ARMS_TO_RUN if ARMS[a]["fmap"] != "none"), ARMS_TO_RUN[0])
probe_cfg = ARMS[probe_arm]

WORKERS = probe_cfg["num_workers"]
data = make_data(probe_cfg, workers=WORKERS)
boot = time.time()
try:
    batch = next(iter(data.train_loader))
except Exception as error:
    # ловим широко нарочно: способов не подняться у spawn много (PicklingError,
    # AttributeError, EOFError, RuntimeError), а ответ на все один и тот же
    print(f"воркеры не поднялись ({type(error).__name__}: {error})\nоткатываюсь на 0")
    WORKERS = 0
    data = make_data(probe_cfg, workers=0)
    batch = next(iter(data.train_loader))
boot = time.time() - boot

print(f"плечо {probe_arm}: train {len(data.train_ds)} "
      f"(негативов {int(data.train_ds.is_negative.sum())}), "
      f"val {len(data.val_ds)} (негативов {int(data.val_ds.is_negative.sum())})")
for key, value in batch.items():
    print(f"  {key:<6} {tuple(value.shape)} {value.dtype} "
          f"[{float(value.min()):+.2f}, {float(value.max()):+.2f}]")
if "fmap" in batch:
    expected = (probe_cfg["bs"], FMAP_USED, probe_cfg["size"] // 8, probe_cfg["size"] // 8)
    assert tuple(batch["fmap"].shape) == expected, f"ждали {expected}"

# Первые батчи в замер не берём. На spawn каждый воркер заново импортирует
# torch/cv2/albumentations — это секунды, и без прогрева замер показал бы, что
# воркеры ВРЕДНЫ. Цена платится один раз на прогон: `persistent_workers=True`
# держит процессы между эпохами.
WARMUP, MEASURE = 5, 25
started, frames = None, 0
for step, item in enumerate(data.train_loader):
    if step == WARMUP:
        started = time.time()
    elif started is not None:
        frames += item["image"].shape[0]
    if step == WARMUP + MEASURE:
        break
rate = frames / (time.time() - started)
need = probe_cfg["bs"] * 3   # грубая оценка: ~3 шага/с на 3070 при 640px
print(f"\nстарт воркеров: {boot:.1f} с (один раз на плечо)")
print(f"даталоадер: {rate:.0f} кадров/с на {WORKERS} воркерах "
      f"(нужно примерно {need}) — " + ("хватает" if rate > need else "УЗКОЕ МЕСТО"))
del data, batch

плечо F1: train 82959 (негативов 2548), val 5664 (негативов 639)
  image  (8, 3, 640, 640) torch.float32 [-2.12, +2.64]
  mask   (8, 1, 640, 640) torch.float32 [+0.00, +1.00]
  label  (8, 1) torch.float32 [+0.00, +1.00]
  fmap   (8, 11, 80, 80) torch.float32 [-4.00, +4.00]

старт воркеров: 2.4 с (один раз на плечо)
даталоадер: 74 кадров/с на 10 воркерах (нужно примерно 24) — хватает


## 5. Модель

In [11]:
def make_norm(kind: str, channels: int) -> nn.Module:
    """BatchNorm или GroupNorm.

    Не косметика. BatchNorm копит бегущие статистики внутри forward, и одно
    переполнение fp16 портит их навсегда: GradScaler защищает веса, но не буферы.
    """
    if kind == "batch":
        return nn.BatchNorm2d(channels)
    if kind == "group":
        return nn.GroupNorm(num_groups=min(32, channels), num_channels=channels)
    raise ValueError(f"неизвестная нормализация: {kind}")


class DWSep(nn.Module):
    """Depthwise-separable 3x3 + BN + GELU, опционально со страйдом 2."""

    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, 3, stride=stride, padding=1, groups=in_ch, bias=False)
        self.bn1 = nn.BatchNorm2d(in_ch)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.act = nn.GELU()

    def forward(self, x):
        x = self.act(self.bn1(self.dw(x)))
        return self.act(self.bn2(self.pw(x)))


class ForensicBranch(nn.Module):
    """Карта 1/8 -> пирамида признаков на страйдах 8, 16, 32.

    Ветка начинается сразу на страйде 8 и оттого почти бесплатна: у SRM-ветки
    два фиксированных фильтра работают на полном разрешении, здесь самый
    дорогой слой считает 80x80. Первый BatchNorm обязателен: каналы карты уже
    приведены к [-1, 1] константами `CHANNEL_RANGE`, но разброс между каналами
    (`latt` против `nz_rel`) остаётся в разы, и без нормировки первые же шаги
    отдали бы всё внимание одному каналу.

    Страйд 4 не трогается сознательно: карта не несёт информации выше 1/8, и
    вливать её на более мелкой сетке нечего — это была бы интерполяция,
    оплаченная FLOPs.
    """

    def __init__(self, in_ch=FMAP_USED, ch8=64, ch16=96, ch32=128):
        super().__init__()
        self.norm0 = nn.BatchNorm2d(in_ch)
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, ch8, 3, padding=1, bias=False),
            nn.BatchNorm2d(ch8), nn.GELU(),
        )
        self.ref8 = DWSep(ch8, ch8)
        self.to16 = DWSep(ch8, ch16, stride=2)
        self.to32 = DWSep(ch16, ch32, stride=2)
        self.out_channels = {8: ch8, 16: ch16, 32: ch32}

    def forward(self, fmap):
        s8 = self.ref8(self.stem(self.norm0(fmap)))
        s16 = self.to16(s8)
        return {8: s8, 16: s16, 32: self.to32(s16)}


class GatedFuse(nn.Module):
    """Остаточное слияние с LayerScale, инициализированной нулём.

    На шаге 0 ветка не влияет ни на что, и модель побитово равна якорю. Это
    убирает главный риск плеча — что новая ветка развалит предобученный энкодер
    в первые же итерации, и вы не поймёте, дело в идее или в старте.

    Мёртвой ветка при этом не становится: градиент по самой `gamma` с нуля
    ненулевой, так что слияние открывается с первого же шага — ровно настолько,
    насколько помогает лоссу. Проверка в 5.1 смотрит и на это.
    """

    def __init__(self, enc_ch: int, aux_ch: int) -> None:
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(enc_ch + aux_ch, enc_ch, 1, bias=False),
            nn.BatchNorm2d(enc_ch), nn.GELU(),
        )
        self.gamma = nn.Parameter(torch.zeros(1, enc_ch, 1, 1))

    def forward(self, enc, aux):
        if aux.shape[-2:] != enc.shape[-2:]:
            aux = F.interpolate(aux, size=enc.shape[-2:], mode="bilinear", align_corners=False)
        return enc + self.gamma * self.proj(torch.cat([enc, aux], 1))


class DecoderBlock(nn.Module):
    """Апсемпл x2 -> конкатенация со скипом -> две свёртки 3x3."""

    def __init__(self, in_ch: int, skip_ch: int, out_ch: int, norm: str) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch + skip_ch, out_ch, 3, padding=1, bias=False)
        self.norm1 = make_norm(norm, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.norm2 = make_norm(norm, out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2.0, mode="nearest")
        if skip is not None:
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:], mode="nearest")
            x = torch.cat([x, skip], dim=1)
        x = self.act(self.norm1(self.conv1(x)))
        return self.act(self.norm2(self.conv2(x)))


class GateHead(nn.Module):
    """Бинарная голова «в кадре есть правка», она же гейт на инференсе."""

    def __init__(self, in_channels: int, dropout: float = 0.2) -> None:
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(in_channels, 1)

    def forward(self, deepest):
        return self.fc(self.drop(torch.flatten(self.pool(deepest), 1)))


def init_module(module: nn.Module) -> None:
    """Kaiming под ReLU/GELU для всего, что учится с нуля.

    Дефолт torch для Conv2d — Kaiming с a=sqrt(5), это gain 0.58 вместо 1.41,
    то есть вдвое тише нужного.
    """
    for m in module.modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.kaiming_uniform_(m.weight, mode="fan_in", nonlinearity="relu")
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

### 5.0 Прогрессивное уточнение решения (arXiv:2607.29156)

Статья «Progressive Decision-Making for Localizing Open-Ended AI-Generated Image
Forgeries» (31.07.2026) предлагает не считать маску одним выстрелом, а трактовать
её как **промежуточное состояние** и обновить `T` раз, глядя на три вещи: улики,
собственную неуверенность и близость к границе. Обновление несимметрично: там,
где модель уже уверена, решение разрешено только усиливать (`max`) или только
ослаблять (`min`); переписывать можно лишь неуверенную середину.

Почему это интересно именно здесь. AIC — гармоническое среднее `Dice` и
`1 - FPR`, и половину метрики держат чистые кадры. Несимметричный гейт смещён
против ложных срабатываний: на уверенно-фоновом пикселе почти весь вес уходит в
ветку `min`, и предложение оператора попадает в ответ только через узкую щель
`M_c`. Полного запрета это не даёт — прогон правила на синтетике показывает, что
отдельные пиксели всё равно растут, — но средний сдвиг на чистом кадре
отрицательный: около `-0.2` логита при `beta = 1` и `-0.8` при `beta = 10`. То
есть механизм давит ровно ту половину метрики, которая здесь дешевле всего
теряется, — но как смещение, а не как запрет.

### Что взято дословно

* `E = Conv1x1(F)`, `C' < C` — проектор улик;
* `U = -P log P - (1-P) log(1-P)` — неуверенность по стартовой карте, считается
  один раз и дальше не меняется;
* `B = exp(-alpha * mean(|dE/dx| + |dE/dy|))` — граничный приор;
* `X_t = Concat(E, P_{t-1}, U, B)`, `P~_t = M(X_t)`;
* `M_s = sigmoid(beta (P - t_high))`, `M_b = sigmoid(beta (t_low - P))`,
  `M_c = 1 - M_s - M_b`, и обновление
  `P_t = M_s max(P_{t-1}, P~_t) + M_c P~_t + M_b min(P_{t-1}, P~_t)`;
* `T = 2`, `t_low = 0.3`, `t_high = 0.8`, `beta = 1`, на инференсе оба порога
  опускаются на `0.2`;
* лоссы `L_deep = sum_t ((t+1)/(T+1))^2 (BCE + Dice)` и `L_aux = BCE(S_aux)`,
  оба с весом 1.

Про `beta = 1` стоит знать заранее. При таком наклоне сигмоиды `M_c` почти
константа: на всём диапазоне `P` от 0 до 1 она держится около `0.12`, а `M_s` и
`M_b` гуляют в пределах `0.31...0.57`. То есть «регионное» разделение у авторов
очень мягкое — это постоянная смесь трёх ветвей, а не роутер. Стартуем с
авторским значением, но если по логу видно, что механизм ничем себя не проявил,
`beta` — первая ручка: при `beta = 10` пороги начинают работать как пороги.

### Три отличия, и почему

**1. Оператор `M` свёрточный, а не Mamba.** Авторская `M` — это SSM, которому
нужен `mamba-ssm` с собственным CUDA-ядром; на Windows он практически не
ставится, а чистый torch-фолбэк материализует тензор `(B, d_inner, L, d_state)` —
на страйде 4 при `bs=16` это под гигабайт на один промежуток, то есть ровно тот
упор в память, который здесь уже превращает шаг в 6.5 секунды. Поэтому `M`
сделан сменным: `kind="conv"` работает сегодня, `kind="mamba"` берёт официальное
ядро, если ноутбук поедет на Linux-GPU.

Это не подмена ради удобства. В абляции авторов (таблица III) вклад собственно
скана не выделен вообще: убрать `L_deep` стоит `-0.064` F1, регионное
ограничение `-0.043`, `L_aux` `-0.045`, проектор улик `-0.033`. Всё это части
**схемы**, а не оператора. Плечо на свёрточной `M` честно проверяет схему; если
она сработает, вопрос про SSM станет отдельным и осмысленным, а не первым.

**2. Обновление идёт в логитах, приращением с нулевой LayerScale.**
`P~_t = sigmoid(logit(P_{t-1}) + gamma * delta)`, `gamma` стартует с нуля.
`max`/`min` от этого не меняются: сигмоида монотонна, порядок сохраняется. Зато
на шаге 0 плечо **побитово** равно якорю, и проверка 5.1 распространяется на
новый блок. Без этого любое расхождение плеч можно было бы списать на сломанный
старт — та же логика, что у `GatedFuse`.

**3. Оператор считается на страйде 4, гейты — на полном разрешении.**
`M_s`/`M_c`/`M_b` поточечные и стоят ноль, а метрика живёт на полном разрешении,
поэтому им там и место. Дорогая часть — сама `M` — остаётся на `160x160`. Это и
есть ответ на «сильно ли вырастут FLOPs»: около `+0.6` строгих GFLOPs, меньше
процента от `88.4`. Тот же блок на полном разрешении стоил бы `+10.0`: с
`convnext_tiny` это формально ещё влезает, а у `pvt_v2_b2` из плеча T (`96.6` из
`100`) свободных гигафлопсов уже нет. Точное число печатает раздел 6, отдельным
слагаемым.

Что действительно вырастет, так это **память под активации**: гейты и
приращение — это восемь полноразмерных тензоров на шаг, два шага, всё в графе
для backward. При `bs=16` на 640 px это сотни мегабайт сверх текущего. Если
эпоха вдруг замедлится в разы, дело в этом, а не в вычислениях: `bs=8` с
`accum_steps=2` даёт тот же эффективный батч.

### Чего ждать

Заголовочный прирост статьи (`0.53 -> 0.64` F1) снят на **кросс-доменном**
протоколе: обучение на классических подделках, тест на невиданных
AI-генерациях. Здесь train и test из одного пайплайна, и там, где улики сильные,
осторожное обновление даёт меньше. Но `plain_l9` — ровно случай слабых улик
(AUC внутри кадра `0.892` при потолке по площади `0.668`), и механизм «не трогай
уверенное, переписывай неуверенное» адресован именно ему.

Порядок чтения результата: сначала `train/refine_gamma` в логе — если он остался
около `1e-3`, блок не пригодился и разница плеч это шум; потом срез по доменам;
и только потом общий AIC.


In [12]:
# ── Прогрессивное уточнение решения (arXiv:2607.29156) ──────────────────────
#
# Отличия от статьи перечислены в markdown выше. Здесь только код; спорить с
# ним надо там.


def binary_entropy(probs, eps: float = 1e-6):
    """Неуверенность U: энтропия бернуллиевского пикселя, максимум ln2 в 0.5.

    Считается в fp32 нарочно. Под `amp="fp16"` логарифм у самых уверенных
    пикселей уходит туда, где шаг сетки fp16 крупнее самого значения, и канал U
    вырождается в ступеньки. Стоит это долей мегабайта: карта на страйде 4.
    """
    p = probs.float().clamp(eps, 1.0 - eps)
    return -(p * p.log() + (1.0 - p) * (1.0 - p).log())


def boundary_prior(evidence, alpha: float):
    """B = exp(-alpha * G), где G — средняя по каналам первая разность улик.

    Единица внутри однородных областей, спад на перепадах. Это множитель «здесь
    решение можно менять смело», а не детектор края: сглаживать или бинаризовать
    G статья не просит.
    """
    e = evidence.float()
    gx = (e[..., :, 1:] - e[..., :, :-1]).abs()
    gy = (e[..., 1:, :] - e[..., :-1, :]).abs()
    grad = (F.pad(gx, (1, 0)) + F.pad(gy, (0, 0, 1, 0))).mean(1, keepdim=True)
    return torch.exp(-alpha * grad)


class DecisionEvidence(nn.Module):
    """E = Conv1x1(F) при C' < C, плюс голова S_aux под L_aux.

    Статья: «channel-wise filtering and recombination without altering spatial
    structure». BN и GELU дописаны здесь, в статье их нет: блок учится с нуля
    рядом с предобученным декодером, и без нормировки L_aux на первых шагах
    перекрикивает основной лосс.
    """

    def __init__(self, in_ch: int, ev_ch: int) -> None:
        super().__init__()
        self.proj = nn.Sequential(
            nn.Conv2d(in_ch, ev_ch, 1, bias=False),
            nn.BatchNorm2d(ev_ch), nn.GELU(),
        )
        self.aux = nn.Conv2d(ev_ch, 1, 1)
        self.out_channels = ev_ch

    def forward(self, feat):
        evidence = self.proj(feat)
        return evidence, self.aux(evidence)


class ConvRefiner(nn.Module):
    """Оператор M на свёртках: два depthwise-блока 7x7 с дилатациями 1 и 3.

    Замена авторской Mamba, обоснование — в markdown. Дилатация 3 на страйде 4
    даёт рецептивное поле 76 px входного кадра; глобального скана это не
    заменяет, поэтому кадровый контекст добирается отдельным гейтом по среднему,
    а не притворяется, что его нет.
    """

    def __init__(self, in_ch: int, hidden: int = 32) -> None:
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden), nn.GELU(),
        )
        self.local = self._block(hidden, dilation=1)
        self.wide = self._block(hidden, dilation=3)
        self.context = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(), nn.Linear(hidden, hidden),
        )
        self.out = nn.Conv2d(hidden, 1, 1)

    @staticmethod
    def _block(ch: int, dilation: int) -> nn.Module:
        return nn.Sequential(
            nn.Conv2d(ch, ch, 7, padding=3 * dilation, dilation=dilation,
                      groups=ch, bias=False),
            nn.BatchNorm2d(ch), nn.GELU(),
            nn.Conv2d(ch, ch, 1, bias=False),
            nn.BatchNorm2d(ch), nn.GELU(),
        )

    def forward(self, x):
        h = self.stem(x)
        h = h + self.local(h)
        h = h + self.wide(h)
        gate = self.context(h.mean(dim=(2, 3)))[:, :, None, None]
        return self.out(h * torch.sigmoid(gate))


class MambaRefiner(nn.Module):
    """Оператор M на официальном ядре mamba-ssm: растровый скан в две стороны.

    Без CUDA-ядра не работает, и это проверяется при сборке, а не на первом
    батче через два часа. Двунаправленность обязательна: односторонний скан по
    строкам делает верх кадра слепым к низу, а подделка не обязана лежать в
    порядке чтения.
    """

    def __init__(self, in_ch: int, hidden: int = 32, d_state: int = 16) -> None:
        super().__init__()
        try:
            from mamba_ssm import Mamba
        except ImportError as error:
            raise ImportError(
                "оператор kind='mamba' требует mamba-ssm с собственным "
                "CUDA-ядром; на Windows он практически не ставится. Ставь "
                "kind='conv' или считай плечо на Linux-GPU."
            ) from error
        self.stem = nn.Sequential(
            nn.Conv2d(in_ch, hidden, 1, bias=False),
            nn.BatchNorm2d(hidden), nn.GELU(),
        )
        self.norm = nn.LayerNorm(hidden)
        self.forward_scan = Mamba(d_model=hidden, d_state=d_state, d_conv=4, expand=2)
        self.reverse_scan = Mamba(d_model=hidden, d_state=d_state, d_conv=4, expand=2)
        self.out = nn.Conv2d(hidden, 1, 1)

    def forward(self, x):
        h = self.stem(x)
        b, c, height, width = h.shape
        seq = self.norm(h.flatten(2).transpose(1, 2))
        seq = self.forward_scan(seq) + self.reverse_scan(seq.flip(1)).flip(1)
        h = h + seq.transpose(1, 2).reshape(b, c, height, width)
        return self.out(h)


class EGRefine(nn.Module):
    """Маска как состояние: T несимметричных обновлений вместо одного выстрела.

    Оператор M считается на страйде 4, а несимметричные гейты M_s/M_c/M_b — на
    полном разрешении: они поточечные и бесплатные, а метрика живёт там же.
    """

    OPERATORS = {"conv": ConvRefiner, "mamba": MambaRefiner}

    def __init__(self, feat_ch: int, ev_ch: int = 16, hidden: int = 32,
                 steps: int = 2, kind: str = "conv", tau_low: float = 0.3,
                 tau_high: float = 0.8, beta: float = 1.0, alpha: float = 1.0,
                 test_offset: float = 0.2) -> None:
        super().__init__()
        if kind not in self.OPERATORS:
            raise ValueError(f"неизвестный оператор {kind!r}; "
                             f"есть {', '.join(self.OPERATORS)}")
        self.evidence = DecisionEvidence(feat_ch, ev_ch)
        self.operator = self.OPERATORS[kind](ev_ch + 3, hidden)
        self.gamma = nn.Parameter(torch.zeros(1))
        self.steps = int(steps)
        self.tau_low, self.tau_high = float(tau_low), float(tau_high)
        self.beta, self.alpha = float(beta), float(alpha)
        self.test_offset = float(test_offset)

    def thresholds(self):
        """На инференсе статья опускает ОБА порога на 0.2."""
        shift = 0.0 if self.training else self.test_offset
        return self.tau_low - shift, self.tau_high - shift

    def forward(self, feat, logits, size):
        evidence, evidence_logits = self.evidence(feat)
        small = evidence.shape[-2:]

        # U и B считаются один раз по стартовому состоянию и дальше не меняются
        prior = boundary_prior(evidence, self.alpha).to(evidence.dtype)
        uncertainty = binary_entropy(
            F.interpolate(torch.sigmoid(logits), size=small, mode="bilinear",
                          align_corners=False)).to(evidence.dtype)
        tau_low, tau_high = self.thresholds()

        state, states = logits, [logits]
        for _ in range(self.steps):
            probs = torch.sigmoid(state)
            probs_small = F.interpolate(probs, size=small, mode="bilinear",
                                        align_corners=False)
            delta = self.operator(
                torch.cat([evidence, probs_small.to(evidence.dtype),
                           uncertainty, prior], dim=1))
            delta = F.interpolate(delta.to(state.dtype), size=size,
                                  mode="bilinear", align_corners=False)

            # Приращение живёт в ЛОГИТАХ: sigmoid монотонна, поэтому max и min
            # из статьи значат ровно то же, что значили бы на вероятностях.
            #
            # gamma приводится к dtype состояния ЯВНО, и это не косметика.
            # autocast правит только свой список операций, умножение в него не
            # входит, поэтому fp32-параметр обычным промоушеном типов поднимает
            # всю цепочку обновления в fp32 на ПОЛНОМ разрешении — а вместе с
            # ней основной лосс и L_deep, которые считаются от `states`.
            diff = self.gamma.to(state.dtype) * delta

            keep_fg = torch.sigmoid(self.beta * (probs - tau_high))
            keep_bg = torch.sigmoid(self.beta * (tau_low - probs))
            revise = (1.0 - keep_fg - keep_bg).clamp(min=0.0)

            # Та же алгебра, что в статье, но записанная через diff:
            #   max(s, s + d) - s == clamp(d, min=0)
            #   min(s, s + d) - s == clamp(d, max=0)
            # Три полноразмерных тензора долой, и при gamma=0 каждое слагаемое
            # равно нулю ТОЧНО, так что проверка 5.1 сходится побитово. Взять
            # исходную форму статьи нельзя: сумма keep_fg + revise + keep_bg
            # в float равна единице лишь приближённо, и старт перестал бы быть
            # тождественным.
            state = state + (keep_fg * diff.clamp(min=0.0)
                             + revise * diff
                             + keep_bg * diff.clamp(max=0.0))
            states.append(state)
        return states, evidence_logits


In [13]:
class Segmenter(nn.Module):
    """U-Net-подобный сегментатор; ветка по форензик-карте включается флагом.

    Флаг обязателен и по умолчанию выключен: иначе плечо A молча собиралось бы с
    веткой внутри и перестало быть якорем. То же и с `refine`: блок уточнения
    приходит словарём из конфига, и `None` означает ровно прежнюю модель.
    """

    FUSE_AT = (8, 16, 32)

    def __init__(self, encoder="convnext_tiny", decoder_channels=(128, 64, 32, 16, 16),
                 norm="batch", pretrained=False, use_fmap=False, fmap_ch=(64, 96, 128),
                 aux_weight=0.0, refine=None, verbose=True):
        super().__init__()
        self.encoder = timm.create_model(encoder, features_only=True,
                                         pretrained=pretrained, in_chans=3)
        self.reductions = list(self.encoder.feature_info.reduction())
        channels = list(self.encoder.feature_info.channels())

        self.use_fmap = use_fmap
        if use_fmap:
            missing = [r for r in self.FUSE_AT if r not in self.reductions]
            if missing:
                raise ValueError(f"у {encoder} нет карт на страйдах {missing}: {self.reductions}")
            self.branch = ForensicBranch(FMAP_USED, *fmap_ch)
            self.fuse = nn.ModuleDict({
                str(r): GatedFuse(channels[self.reductions.index(r)],
                                  self.branch.out_channels[r])
                for r in self.FUSE_AT
            })

        skip_at = dict(zip(self.reductions[:-1], channels[:-1]))
        steps = int(math.log2(self.reductions[-1]))
        self.skip_reductions, blocks = [], []
        in_ch, reduction = channels[-1], self.reductions[-1]
        for i in range(steps):
            reduction //= 2
            skip_ch = skip_at.get(reduction, 0)
            self.skip_reductions.append(reduction if skip_ch else None)
            blocks.append(DecoderBlock(in_ch, skip_ch, decoder_channels[i], norm))
            in_ch = decoder_channels[i]
        self.blocks = nn.ModuleList(blocks)
        self.head = nn.Conv2d(in_ch, 1, 3, padding=1)
        self.gate = GateHead(channels[-1])
        self.aux_weight = aux_weight
        self.aux4 = nn.Conv2d(decoder_channels[2], 1, 1) if aux_weight > 0 else None

        # Улики берутся с того же отвода страйда 4, что и aux4. Это самое
        # «мезоскопическое» место сети: семантика уже пришла сверху по декодеру,
        # форензик-ветка влита на 8/16/32 и тоже дошла, а разрешение ещё не
        # разбавлено двумя апсемплами.
        self.refine = EGRefine(decoder_channels[2], **refine) if refine else None

        # инициализируем ТОЛЬКО то, что учится с нуля: энкодер предобучен
        for part in (self.blocks, self.head, self.gate):
            init_module(part)
        if self.aux4 is not None:
            init_module(self.aux4)
        if use_fmap:
            init_module(self.branch)
            init_module(self.fuse)
            for r in self.FUSE_AT:      # init_module прошёлся и по gamma — вернуть ноль
                nn.init.zeros_(self.fuse[str(r)].gamma)
        if self.refine is not None:
            init_module(self.refine)
            nn.init.zeros_(self.refine.gamma)

        if verbose:
            n = sum(p.numel() for p in self.parameters()) / 1e6
            without = [i for i, r in enumerate(self.skip_reductions) if r is None]
            print(f"  {encoder}: страйды {self.reductions}, каналы {channels}")
            print(f"  блоков декодера без skip: {without or 'нет'}")
            print(f"  ветка карты={use_fmap} {fmap_ch if use_fmap else ''}, "
                  f"aux={aux_weight}, {n:.1f}M параметров")
            if self.refine is not None:
                print(f"  уточнение: оператор {refine.get('kind', 'conv')}, "
                      f"T={self.refine.steps}, улик {self.refine.evidence.out_channels}")

    def forward(self, x, fmap=None):
        size = x.shape[-2:]
        feats = list(self.encoder(x))

        if self.use_fmap:
            if fmap is None:
                # так модель остаётся вызываемой одним тензором: этого требует
                # счётчик FLOPs, и на нуле ветка честно считается, а не пропускается
                fmap = x.new_zeros(x.shape[0], FMAP_USED, size[0] // 8, size[1] // 8)
            branch = self.branch(fmap.to(x.dtype))
            for r in self.FUSE_AT:
                index = self.reductions.index(r)
                feats[index] = self.fuse[str(r)](feats[index], branch[r])

        by_red = dict(zip(self.reductions, feats))
        out, aux, feat4 = feats[-1], None, None
        for i, (block, r) in enumerate(zip(self.blocks, self.skip_reductions)):
            out = block(out, by_red.get(r) if r else None)
            if i == 2:
                feat4 = out
                if self.aux4 is not None:
                    aux = self.aux4(out)

        logits = self.head(out)
        if logits.shape[-2:] != size:
            logits = F.interpolate(logits, size=size, mode="bilinear", align_corners=False)
        result = {"logits": logits, "cls_logits": self.gate(feats[-1])}

        if self.refine is not None:
            states, evidence_logits = self.refine(feat4, logits, size)
            # наружу уходит ПОСЛЕДНЕЕ состояние: валидация, калибровка порогов и
            # сабмит обязаны смотреть на то же, что и лосс
            result["logits"] = states[-1]
            if self.training:
                result["states"] = states
                result["evidence_logits"] = F.interpolate(
                    evidence_logits, size=size, mode="bilinear", align_corners=False)

        if self.training and aux is not None:
            result["aux_logits"] = F.interpolate(aux, size=size, mode="bilinear",
                                                 align_corners=False)
        return result


def build_model(cfg, pretrained=True, verbose=True):
    return Segmenter(
        encoder=cfg["encoder"],
        decoder_channels=tuple(cfg["decoder_channels"]),
        norm=cfg["norm"],
        pretrained=pretrained,
        use_fmap=cfg["fmap"] != "none",
        fmap_ch=tuple(cfg["fmap_ch"]),
        aux_weight=cfg["aux_weight"],
        refine=cfg.get("refine") or None,
        verbose=verbose,
    )


### 5.1 Проверка: на шаге 0 модель с веткой равна модели без ветки

Если это не так, любое расхождение плеч можно объяснить сломанным стартом, а не
идеей. Проверка грубая нарочно: веса ветки умножаются на 50, и выход обязан не
дрогнуть **побитово**.

In [14]:
def check_zero_start(size: int = 128, refine: dict | None = None) -> None:
    """Всё, что учится с нуля поверх якоря, обязано молчать на шаге 0.

    Проверка грубая нарочно: веса новых блоков умножаются на 50, и выход обязан
    не дрогнуть ПОБИТОВО. Блок уточнения проверяется тем же способом и на том же
    основании — если он сдвигает выход до первого шага, разницу плеч можно будет
    объяснить сломанным стартом, а не идеей статьи.
    """
    model = Segmenter(use_fmap=True, aux_weight=0.4, refine=refine, verbose=False).eval()
    x = torch.randn(2, 3, size, size)
    fmap = torch.randn(2, FMAP_USED, size // 8, size // 8)
    with torch.no_grad():
        before = model(x, fmap)["logits"]
        for p in model.branch.parameters():
            p.mul_(50.0)
        for r in model.FUSE_AT:
            for p in model.fuse[str(r)].proj.parameters():
                p.mul_(50.0)
        if model.refine is not None:
            for p in model.refine.operator.parameters():
                p.mul_(50.0)
        after = model(x, fmap)["logits"]
    delta = (before - after).abs().max().item()
    what = "ветка и уточнение" if refine else "ветка"
    print(f"{what} отключены на старте: max|Δ| = {delta:.2e}",
          "OK" if delta == 0 else "СЛОМАНО")

    # и вторая половина того же: гейт открывается градиентом, а не остаётся мёртвым
    model.train()
    model(x, fmap)["logits"].sum().backward()
    grad = model.fuse["8"].gamma.grad
    print(f"градиент по gamma ветки на шаге 0: {grad.abs().max().item():.2e}",
          "OK" if grad.abs().max() > 0 else "ВЕТКА МЁРТВАЯ")
    if model.refine is not None:
        grad = model.refine.gamma.grad
        print(f"градиент по gamma уточнения на шаге 0: {grad.abs().max().item():.2e}",
              "OK" if grad.abs().max() > 0 else "БЛОК МЁРТВЫЙ")


check_zero_start()
check_zero_start(refine=REFINE)


ветка отключены на старте: max|Δ| = 0.00e+00 OK
градиент по gamma ветки на шаге 0: 1.34e+03 OK
ветка и уточнение отключены на старте: max|Δ| = 0.00e+00 OK
градиент по gamma ветки на шаге 0: 1.12e+03 OK
градиент по gamma уточнения на шаге 0: 5.08e+07 OK


## 6. Бюджет FLOPs — до обучения

In [15]:
def budget_of(cfg):
    """Вердикт по лимиту. Считается на `meta`: без арифметики и без памяти,
    поэтому карту не занимает и стоит доли секунды."""
    with torch.device("meta"):
        net = build_model(cfg, pretrained=False, verbose=False)
    return budget.check(net, cfg["size"])


for arm in ARMS_TO_RUN:
    cfg = ARMS[arm]
    verdict = budget_of(cfg)

    # Раскладка по слагаемым, а не одно число: вопрос «сколько стоит новая
    # надстройка» задаётся каждый раз, и отвечать на него вычитанием в уме —
    # верный способ однажды ошибиться в пользу приятного ответа.
    naked = budget_of({**cfg, "fmap": "none", "refine": None}).gflops
    with_branch = budget_of({**cfg, "refine": None}).gflops
    parts = []
    if cfg["fmap"] != "none":
        parts.append(f"ветка карты {with_branch - naked:+.2f}")
    if cfg.get("refine"):
        parts.append(f"уточнение {verdict.gflops - with_branch:+.2f}")
    extra = ("; из них " + ", ".join(parts)) if parts else ""

    print(f"[{arm}] {verdict.text}{extra}")
    if not verdict.within_limit:
        raise SystemExit(f"плечо {arm} вне бюджета — правь size, fmap_ch или refine")


[F1] 97.2 из 100 GFLOPs на изображение (640px) — в бюджете; из них ветка карты +1.18
[F0] 97.2 из 100 GFLOPs на изображение (640px) — в бюджете; из них ветка карты +1.18


## 7. Лосс, валидация

In [16]:
# Слагаемые лосса живут в `s3_losses.py`, а не здесь: ячейку не покрыть тестом,
# а именно в этих функциях ошибка не падает, а тихо портит обучение. Проверки —
# `tests/test_loss_terms.py`. Здесь остаётся то, что завязано на глобали
# ноутбука: валидация (DEVICE, AMP) и группы оптимизатора.
import importlib

import s3_losses

importlib.reload(s3_losses)          # правки в модуле без рестарта ядра
from s3_losses import compute_loss, soft_dice   # noqa: E402,F401


N_BINS = 256


@torch.no_grad()
def validate(net, loader):
    net.eval()
    acc = aic.AICAccumulator(n_bins=N_BINS)
    for batch in loader:
        images = batch["image"].to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        masks = batch["mask"].to(DEVICE, non_blocking=True)
        fmap = batch["fmap"].to(DEVICE, non_blocking=True) if "fmap" in batch else None
        with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            out = net(images, fmap)
        probs = torch.sigmoid(out["logits"].float())
        cls = torch.sigmoid(out["cls_logits"].float()).reshape(-1)

        b = probs.shape[0]
        flat = probs.clamp(0, 1).reshape(b, -1)
        idx = (flat * N_BINS).long().clamp_(max=N_BINS - 1)
        offset = torch.arange(b, device=idx.device).unsqueeze(1) * N_BINS
        gt = masks.reshape(b, -1) > 0.5
        hist_all = torch.bincount((idx + offset).reshape(-1),
                                  minlength=b * N_BINS).reshape(b, N_BINS)
        hist_gt = torch.bincount((idx + offset)[gt], minlength=b * N_BINS).reshape(b, N_BINS)

        acc.update_hist(hist_all.cpu().numpy(), hist_gt.cpu().numpy(),
                        gt.sum(1).cpu().numpy(),
                        np.full(b, flat.shape[1], dtype=np.int64), cls.cpu().numpy())
    return acc


def build_optimizer(model, cfg):
    """Пять групп, а не три. Порядок проверок важен: сначала определяется
    ПРИНАДЛЕЖНОСТЬ, и только потом decay. Если сначала спросить `p.ndim <= 1`,
    все LayerNorm с LayerScale предобученного ConvNeXt уедут на полный lr вместо
    `encoder_lr`, а у ConvNeXt LayerScale задаёт вклад каждого блока — втрое
    завышенный lr на нём расшатывает предобучение.

    Блок уточнения (`refine.`) сюда специально не добавлен отдельной группой: он
    учится с нуля, как декодер, и живёт на том же `lr`.
    """
    groups = {"enc": [], "enc_nd": [], "fmap": [], "dec": [], "dec_nd": []}
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        no_decay = p.ndim <= 1 or name.endswith("gamma")
        if name.startswith(("branch.", "fuse.")):
            groups["fmap"].append(p)                   # учится с нуля -> полный lr
        elif "encoder." in name:
            groups["enc_nd" if no_decay else "enc"].append(p)
        else:
            groups["dec_nd" if no_decay else "dec"].append(p)

    return torch.optim.AdamW([
        {"params": groups["enc"], "lr": cfg["encoder_lr"], "weight_decay": cfg["weight_decay"]},
        {"params": groups["enc_nd"], "lr": cfg["encoder_lr"], "weight_decay": 0.0},
        {"params": groups["fmap"], "lr": cfg["fmap_lr"], "weight_decay": cfg["weight_decay"]},
        {"params": groups["dec"], "lr": cfg["lr"], "weight_decay": cfg["weight_decay"]},
        {"params": groups["dec_nd"], "lr": cfg["lr"], "weight_decay": 0.0},
    ])


## 8. Обучение

`train_arm` считает одно плечо целиком и кладёт его в свой прогон. Ячейка ниже
зовёт её по списку `ARMS_TO_RUN`, так что «Run All» посчитает всю серию за один
заход. Прерванный ноутбук продолжится с последней сохранённой эпохи каждого
плеча: `resume` смотрит на `ckpt/last.pt` внутри прогона, а не на состояние ядра.

`train/fmap_gamma` в логе — не украшение. Это максимум модуля LayerScale по трём
слияниям: он стартует ровно с нуля, и если к концу обучения так и остался около
`1e-3`, ветка не пригодилась, и любая разница плеч — шум. Смотреть на него надо
раньше, чем на AIC.

In [17]:
def gpu_memory_report() -> str:
    """Сколько на карте свободно и кто это занял.

    `torch.cuda.mem_get_info` учитывает ЧУЖИЕ процессы, в отличие от
    `torch.cuda.memory_allocated`: тот видит только свои аллокации и на занятой
    карте покажет почти ноль ровно перед OOM — то есть соврёт в самый нужный
    момент.
    """
    if DEVICE.type != "cuda":
        return "устройство не CUDA"
    free, total = torch.cuda.mem_get_info()
    lines = [f"карта: {free / 1e9:.2f} ГБ свободно из {total / 1e9:.2f}"]

    # чужие процессы — главная причина, по которой свободного меньше, чем ждёшь
    try:
        import subprocess

        raw = subprocess.run(
            ["nvidia-smi", "--query-compute-apps=pid,used_memory",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10, check=True).stdout.strip()
        mine = str(os.getpid())
        others = []
        for row in raw.splitlines():
            pid, _, used = row.partition(",")
            pid, used = pid.strip(), used.strip()
            try:
                amount = f"{int(used) / 1024:.2f} ГБ"
            except ValueError:
                # на Windows драйвер WDDM объём по процессам не отдаёт вовсе
                amount = "объём не сообщается"
            tag = "этот процесс" if pid == mine else "ЧУЖОЙ"
            others.append(f"    pid {pid}: {amount}  ({tag})")
        if others:
            lines.append("  процессы на карте:")
            lines.extend(others)
    except Exception as error:
        lines.append(f"  (список процессов не получить: {type(error).__name__})")
    return "\n".join(lines)


def probe_memory(model, cfg) -> tuple[bool, float]:
    """Один шаг вперёд-назад на случайных тензорах: `(влезло, пик в ГБ)`.

    Делается ДО построения загрузчика намеренно. Воркеры на spawn поднимаются
    около восемнадцати секунд, датасет читает индекс, и узнавать о нехватке
    памяти после всего этого — значит платить за него на каждой попытке. Плюс
    настоящий OOM прилетает изнутри attention энкодера и выглядит как поломка
    модели, хотя причина в том, что карту занял кто-то другой.
    """
    if DEVICE.type != "cuda":
        return True, 0.0
    size, bs = int(cfg["size"]), int(cfg["bs"])
    images = fmap = batch = loss = None

    # Состояние генератора сохраняется и возвращается. Проба тратит случайные
    # числа на свои тензоры, и без этого плечо обучалось бы на другом потоке
    # RNG, чем якорь: разошлись бы и dropout в гейте, и порядок сэмплера.
    # Тихая потеря сопоставимости ровно там, где мы её и меряем.
    rng_cpu = torch.get_rng_state()
    rng_cuda = torch.cuda.get_rng_state_all()
    try:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        images = torch.randn(bs, 3, size, size, device=DEVICE).to(
            memory_format=torch.channels_last)
        fmap = (torch.randn(bs, FMAP_USED, size // 8, size // 8, device=DEVICE)
                if cfg["fmap"] != "none" else None)
        batch = {
            "mask": (torch.rand(bs, 1, size, size, device=DEVICE) > 0.7).float(),
            "label": torch.ones(bs, 1, device=DEVICE),
        }
        # термы серии L тянут свои цели, и память под них тоже надо померить
        if float(cfg.get("w_comp", 0.0)) > 0:
            batch["comp"] = torch.randint(0, 4, (bs, 1, size, size),
                                          device=DEVICE, dtype=torch.int16)
        if float(cfg.get("w_far", 0.0)) > 0:
            batch["dist"] = torch.rand(bs, 1, size, size, device=DEVICE)

        with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            loss, _ = compute_loss(model(images, fmap), batch, cfg, progress=1.0)
        loss.backward()
        return True, torch.cuda.max_memory_allocated() / 1e9
    except torch.cuda.OutOfMemoryError:
        return False, float("nan")
    except RuntimeError as error:
        # не во всех сборках torch OOM приходит отдельным классом
        if "out of memory" not in str(error).lower():
            raise
        return False, float("nan")
    finally:
        model.zero_grad(set_to_none=True)
        del images, fmap, batch, loss
        torch.cuda.empty_cache()
        torch.set_rng_state(rng_cpu)
        torch.cuda.set_rng_state_all(rng_cuda)


def preflight_memory(model, cfg, arm: str) -> dict:
    """Проверить память до обучения. Вернуть конфиг — возможно, с другим батчем.

    Батч по умолчанию НЕ трогается, и это осознанно. Якорь серии
    `s3-T-mitb2-native640` считался на `bs=8`, а BatchNorm декодера копит
    статистику по микробатчу: плечо на `bs=4` отвечает уже на другой вопрос,
    сколько ни правь `accum_steps`. Молча разменять сопоставимость на то, чтобы
    прогон стартовал, — худший из возможных вариантов, поэтому по умолчанию
    ячейка останавливается и говорит, что делать.

    `allow_batch_refit=True` в конфиге разрешает подбор — с громким
    предупреждением в лог прогона.
    """
    print(gpu_memory_report())
    ok, peak = probe_memory(model, cfg)
    if ok:
        free, _ = (torch.cuda.mem_get_info() if DEVICE.type == "cuda" else (0, 0))
        print(f"  пробный шаг при bs={cfg['bs']}: пик {peak:.2f} ГБ, "
              f"свободно после него {free / 1e9:.2f} ГБ")
        return cfg

    if not cfg.get("allow_batch_refit", False):
        raise RuntimeError(
            f"[{arm}] не хватает памяти на bs={cfg['bs']} при {cfg['size']}px.\n"
            f"{gpu_memory_report()}\n"
            "Это НЕ ошибка модели: прогон s3-T-mitb2-native640_long на этой же\n"
            "карте занимал 15.26 ГБ при bs=16, то есть примерно 7-8 ГБ при bs=8.\n"
            "Если в списке выше есть чужие процессы — освободите карту, это\n"
            "обычно забытое ядро прошлого запуска.\n"
            "Батч я не трогаю специально: якорь s3-T-mitb2-native640 считался на\n"
            "bs=8, а BatchNorm декодера копит статистику по микробатчу — плечо на\n"
            "bs=4 с якорем уже не сравнить. Если сопоставимость не нужна, поставьте\n"
            "allow_batch_refit=True в конфиге плеча."
        )

    target = int(cfg["bs"]) * int(cfg["accum_steps"])
    bs = int(cfg["bs"])
    while bs > 1:
        bs //= 2
        print(f"  bs={bs * 2} не влез, пробую bs={bs}")
        ok, peak = probe_memory(model, {**cfg, "bs": bs})
        if ok:
            cfg = {**cfg, "bs": bs, "accum_steps": max(1, target // bs)}
            print(f"  ВНИМАНИЕ: батч подобран — bs={bs}, accum={cfg['accum_steps']} "
                  f"(эффективный {bs * cfg['accum_steps']}, пик {peak:.2f} ГБ).")
            print(f"  Плечо больше НЕ сравнимо с якорем напрямую: у того bs=8.")
            return cfg
    raise RuntimeError(
        f"[{arm}] не влезает даже bs=1.\n{gpu_memory_report()}\n"
        "Карту занял кто-то другой, либо разрешение слишком велико для неё."
    )


MASK_GRID = tuple(np.round(np.arange(0.05, 0.96, 0.025), 4).tolist())
CLS_GRID = (0.0, 0.2, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95)
AREA_GRID = (0.0,)


def train_arm(arm: str):
    """Одно плечо: данные, модель, обучение, прогон на диске."""
    cfg = dict(ARMS[arm])          # копия: проверка памяти может поправить батч
    verdict = budget_of(cfg)
    torch.manual_seed(cfg["seed"])
    np.random.seed(cfg["seed"])

    # Модель и проверка памяти идут ДО загрузчика. Воркеры на spawn поднимаются
    # около восемнадцати секунд, и узнавать о нехватке памяти после этого значит
    # платить за них на каждой попытке; а сам OOM прилетает изнутри attention
    # энкодера и выглядит как поломка модели, хотя карту занял кто-то другой.
    model = build_model(cfg).to(DEVICE, memory_format=torch.channels_last)

    # Дотюн стартует с весов другого прогона, а не с ImageNet. Проверка на
    # собственный `last.pt` обязательна: `init_from` нужен только на ПЕРВОМ
    # запуске, иначе прерванный дотюн при продолжении откатился бы к исходным
    # весам и потерял всё, что успел выучить.
    if cfg.get("init_from") and not (ws.runs / cfg["name"] / "ckpt" / "last.pt").exists():
        donor = aic.Run.open(ws.runs / cfg["init_from"])
        state = donor.load_state("best.pt", map_location="cpu")
        key = next(k for k in ("ema", "model") if state.get(k))
        model.load_state_dict(state[key], strict=True)
        print(f"  старт с весов {cfg['init_from']}/ckpt/best.pt (ключ {key!r})")

    cfg = preflight_memory(model, cfg, arm)

    data = make_data(cfg, workers=WORKERS)
    optimizer = build_optimizer(model, cfg)

    steps_per_epoch = len(data.train_loader) // cfg["accum_steps"]
    total_steps = max(1, steps_per_epoch * cfg["epochs"])
    warmup = int(cfg["warmup_frac"] * total_steps)

    def lr_lambda(step):
        if step < warmup:
            return (step + 1) / max(1, warmup)
        progress = (step - warmup) / max(1, total_steps - warmup)
        floor = cfg["min_lr_factor"]
        return floor + (1 - floor) * 0.5 * (1 + math.cos(math.pi * min(progress, 1.0)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    ema = torch.optim.swa_utils.AveragedModel(
        model, avg_fn=lambda avg, cur, _: cfg["ema_decay"] * avg + (1 - cfg["ema_decay"]) * cur
    )

    run = aic.Run.create(ws.runs, cfg["name"], resume=cfg["resume"])
    run.save_snapshot({**cfg, "arm": arm,
                       "decoder_channels": list(cfg["decoder_channels"]),
                       "fmap_ch": list(cfg["fmap_ch"]),
                       "crop_scale": list(cfg["crop_scale"]),
                       "fmap_drop": list(cfg["fmap_drop"]),
                       "fmap_channels": [FMAP_CHANNELS[i] for i in FMAP_KEEP],
                       "num_workers": WORKERS,
                       "probe_auc": probe_report,
                       "gflops": round(verdict.gflops, 1)})
    run.info(f"[{arm}] {cfg['name']}: карта={cfg['fmap']}, {verdict.text}")

    start_epoch, seen_total, best_aic, best_result = 0, 0, -1.0, None
    if cfg["resume"] and (run.dir / "ckpt" / "last.pt").exists():
        state = run.load_state("last.pt", map_location=DEVICE)
        model.load_state_dict(state["model"])
        ema.module.load_state_dict(state["ema"])
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        if state.get("scaler"):        # под bf16 GradScaler выключен и отдаёт {}
            scaler.load_state_dict(state["scaler"])
        seen_total = int(state.get("samples", (int(state["epoch"]) + 1) * cfg["epoch_size"]))
        start_epoch = seen_total // cfg["epoch_size"]
        best_aic = max(float(state.get("best_aic", -1.0)),
                       float(run.summary.get("best_aic", -1.0)))
        run.info(f"ПРОДОЛЖАЮ: эпоха {start_epoch} из {cfg['epochs']}, "
                 f"показов {seen_total}, лучший AIC {best_aic:.4f}")

    for epoch in range(start_epoch, cfg["epochs"]):
        model.train()
        started, skipped, total_loss, seen = time.time(), 0, 0.0, 0
        term_sums: dict[str, float] = {}
        optimizer.zero_grad(set_to_none=True)

        for step, batch in enumerate(data.train_loader):
            batch = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
            images = batch["image"].to(memory_format=torch.channels_last)

            # доля пройденного обучения: от неё идёт разогрев гейта
            progress = min(1.0, (epoch * steps_per_epoch + (step + 1) / cfg["accum_steps"])
                           / max(total_steps, 1))
            with torch.amp.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                loss, parts = compute_loss(model(images, batch.get("fmap")), batch,
                                           cfg, progress)

            scaler.scale(loss / cfg["accum_steps"]).backward()
            if (step + 1) % cfg["accum_steps"] == 0:
                if cfg["grad_clip"]:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
                before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                skipped += int(scaler.get_scale() < before)   # шаг выброшен по inf/nan
                optimizer.zero_grad(set_to_none=True)
                ema.update_parameters(model)
                scheduler.step()

            total_loss += float(loss.detach()) * images.size(0)
            seen += images.size(0)
            # синхронизация уже произошла строкой выше, поэтому слагаемые
            # снимаются почти бесплатно — а без них терм с неудачным весом
            # неотличим от терма, которому нечего лечить
            for name, value in parts.items():
                term_sums[name] = term_sums.get(name, 0.0) + float(value) * images.size(0)

            # Прогресс ВНУТРИ эпохи. Без него эпоха — это десять с лишним минут
            # полной тишины, и «медленно» неотличимо от «повисло»: остаётся
            # гадать по nvidia-smi, а он после смерти процесса уже ничего не
            # помнит. Синхронизация по `loss` всё равно происходит строкой выше,
            # так что печать раз в 50 шагов ничего не стоит.
            if step % 1000 == 0:
                elapsed = time.time() - started
                left = (len(data.train_loader) - step - 1) * elapsed / max(step + 1, 1)
                peak = (torch.cuda.max_memory_allocated() / 1e9
                        if DEVICE.type == "cuda" else 0.0)
                print(f"  [{arm}] эпоха {epoch}: шаг {step + 1}/{len(data.train_loader)}"
                      f" | {seen / max(elapsed, 1e-9):5.1f} кадр/с"
                      f" | осталось ~{left / 60:4.1f} мин"
                      f" | пик {peak:.2f} ГБ", flush=True)

        seen_total += cfg["epoch_size"]
        acc = validate(ema.module, data.val_loader)
        tuned = acc.best(list(MASK_GRID), list(CLS_GRID), list(AREA_GRID))

        run.log(epoch, {
            "samples": seen_total,                    # ось для сравнения прогонов
            "train/loss": total_loss / max(1, seen),
            "train/lr": optimizer.param_groups[0]["lr"],
            "train/skipped_steps": skipped,
            "train/fmap_gamma": (float(max(model.fuse[str(r)].gamma.abs().max()
                                           for r in model.FUSE_AT))
                                 if cfg["fmap"] != "none" else 0.0),
            "train/refine_gamma": (float(model.refine.gamma.abs().max())
                                   if model.refine is not None else 0.0),
            **{f"train/term_{name}": value / max(1, seen)
               for name, value in term_sums.items()},
            "val/aic_tuned": tuned.aic,
            "val/dice_tuned": tuned.dice_pos,
            "val/fpr_tuned": tuned.fpr_neg,
            "val/best_thr": tuned.mask_threshold,
            "val/best_cls_thr": tuned.cls_threshold,
            "epoch_time_s": round(time.time() - started, 1),
            "gpu_gb": (round(torch.cuda.max_memory_allocated() / 1e9, 2)
                       if DEVICE.type == "cuda" else 0.0),
        })
        run.info(f"эпоха {epoch}: {tuned} | пропущено шагов {skipped}")
        if skipped > steps_per_epoch * 0.05:
            run.info("  ВНИМАНИЕ: пропущено >5% шагов — переполнения fp16, ставь amp='bf16'")

        if tuned.aic > best_aic:
            best_aic, best_result = tuned.aic, tuned
            run.save_state({"model": model.state_dict(), "ema": ema.module.state_dict(),
                            "epoch": epoch, "samples": seen_total, "best_aic": best_aic,
                            "cfg": cfg}, "best.pt")
            run.save_eval(acc, data.val_df)    # val_df несёт колонку subsource
            run.info(f"  новый лучший AIC {best_aic:.4f} -> ckpt/best.pt")

        run.save_state({"model": model.state_dict(), "ema": ema.module.state_dict(),
                        "optimizer": optimizer.state_dict(),
                        "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                        "epoch": epoch, "samples": seen_total, "best_aic": best_aic,
                        "cfg": cfg}, "last.pt")

    run.save_summary({"best_aic": best_aic,
                      "best": (best_result.as_dict() if best_result is not None
                               else run.summary.get("best")),
                      "samples": seen_total,
                      "gflops": round(verdict.gflops, 1),
                      "within_limit": verdict.within_limit})
    run.close()

    del model, ema, optimizer, data
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return run

In [18]:
import os

print("os.cpu_count():", os.cpu_count())

if hasattr(os, "sched_getaffinity"):
    print("available CPUs:", len(os.sched_getaffinity(0)))

os.cpu_count(): 10
available CPUs: 10


In [19]:
import torch
import time
import os

print("GPU:", torch.cuda.get_device_name())
print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("cuDNN:", torch.backends.cudnn.version())
print("BF16:", torch.cuda.is_bf16_supported())

print("os.cpu_count:", os.cpu_count())
if hasattr(os, "sched_getaffinity"):
    print("CPU affinity:", len(os.sched_getaffinity(0)))

GPU: Tesla V100-SXM2-16GB
Torch: 2.6.0+cu124
CUDA: 12.4
cuDNN: 90100
BF16: True
os.cpu_count: 10
CPU affinity: 10


In [20]:
print(torch.cuda.get_device_capability())
print(torch.cuda.is_bf16_supported())
print(torch.cuda.is_bf16_supported(including_emulation=False))

(7, 0)
True
False


In [21]:
# import time
# import pandas as pd


# def bench_loader(arm, workers, warmup=5, measure=25):
#     cfg = ARMS[arm]
#     data = make_data(cfg, workers=workers)

#     it = iter(data.train_loader)

#     for _ in range(warmup):
#         next(it)

#     frames = 0
#     t0 = time.perf_counter()

#     for _ in range(measure):
#         batch = next(it)
#         frames += len(batch["image"])

#     dt = time.perf_counter() - t0

#     return frames / dt


# rows = []

# for arm in ["A", "P", "N"]:
#     for workers in [0, 1, 2, 4, 6, 8, 10]:
#         rate = bench_loader(arm, workers)

#         print(
#             f"{arm=} {workers=:2d}: "
#             f"{rate:.1f} img/s"
#         )

#         rows.append({
#             "arm": arm,
#             "workers": workers,
#             "img_s": rate,
#         })

# pd.DataFrame(rows)

In [22]:
import numpy as np
from threadpoolctl import threadpool_info
import pprint

np.show_config()
pprint.pp(threadpool_info())

Build Dependencies:
  blas:
    detection method: pkgconfig
    found: true
    include directory: /opt/_internal/cpython-3.11.14/lib/python3.11/site-packages/scipy_openblas64/include
    lib directory: /opt/_internal/cpython-3.11.14/lib/python3.11/site-packages/scipy_openblas64/lib
    name: scipy-openblas
    openblas configuration: OpenBLAS 0.3.31.188.0  USE64BITINT DYNAMIC_ARCH NO_AFFINITY
      Haswell MAX_THREADS=64
    pc file directory: /project/.openblas
    version: 0.3.31.188.0
  lapack:
    detection method: pkgconfig
    found: true
    include directory: /opt/_internal/cpython-3.11.14/lib/python3.11/site-packages/scipy_openblas64/include
    lib directory: /opt/_internal/cpython-3.11.14/lib/python3.11/site-packages/scipy_openblas64/lib
    name: scipy-openblas
    openblas configuration: OpenBLAS 0.3.31.188.0  USE64BITINT DYNAMIC_ARCH NO_AFFINITY
      Haswell MAX_THREADS=64
    pc file directory: /project/.openblas
    version: 0.3.31.188.0
Compilers:
  c:
    commands: 

In [ ]:
runs = {}
for arm in ARMS_TO_RUN:
    print(f"\n{'=' * 70}\n=== плечо {arm}: {ARMS[arm]['name']}\n{'=' * 70}")
    runs[arm] = train_arm(arm)
    print(f"плечо {arm} готово: {runs[arm].summary.get('best')}")


=== плечо F1: s3-F1-fullframe-ft640


  pvt_v2_b2: страйды [4, 8, 16, 32], каналы [64, 128, 320, 512]
  блоков декодера без skip: [3, 4]
  ветка карты=True (64, 96, 128), aux=0.4, 26.7M параметров
  старт с весов s3-T-mitb2-native640_long/ckpt/best.pt (ключ 'ema')
карта: 16.42 ГБ свободно из 16.93
  процессы на карте:
    pid 6117: 0.47 ГБ  (этот процесс)
  пробный шаг при bs=8: пик 7.48 ГБ, свободно после него 16.35 ГБ
[21:52:40] [F1] s3-F1-fullframe-ft640: карта=native, 97.2 из 100 GFLOPs на изображение (640px) — в бюджете
  [F1] эпоха 0: шаг 1/3000 |   3.2 кадр/с | осталось ~123.2 мин | пик 7.63 ГБ
  [F1] эпоха 0: шаг 1001/3000 |  34.8 кадр/с | осталось ~ 7.7 мин | пик 7.95 ГБ
  [F1] эпоха 0: шаг 2001/3000 |  34.7 кадр/с | осталось ~ 3.8 мин | пик 7.95 ГБ
[22:05:48] эпоха 0: AIC=0.9227 (Dice_pos=0.8634, FPR_neg=0.0094) @ thr=0.375 cls=0.500 min_area=0.000 | pos=5025 neg=639 | пропущено шагов 5
[22:05:49]   новый лучший AIC 0.9227 -> ckpt/best.pt
  [F1] эпоха 1: шаг 1/3000 |   2.9 кадр/с | осталось ~138.2 мин | пик 7.95 

## 9. Вердикт: против якоря и против своей пары

Из двух вердиктов решающий — **второй**. Первый отвечает на вопрос «полезна ли
ветка вообще» и смешивает два эффекта: новую ёмкость и новый признак. Второй
изолирует ровно порядок вычислений: у плеч N и P одинаковая ветка, одинаковое
число каналов, одинаковый бюджет FLOPs и одинаковые данные — различается только
то, до ресайза посчитана карта или после.

`train_sigma=0.0035` — шум обучения этой линии, снятый на парах сидов в серии
`s2`. Дельта меньше него ничего не значит, сколько бы ни было знаков после
запятой; ради этого в `ARMS` и лежат плечи `P7`/`N7`.

In [ ]:
def open_run(name: str):
    """Прогон по имени, откуда бы он ни взялся — из этого запуска или из прошлого."""
    return aic.Run.open(ws.runs / name) if (ws.runs / name).exists() else None


def verdict_against(arm: str, ref_name: str, title: str) -> None:
    own, ref = open_run(ARMS[arm]["name"]), open_run(ref_name)
    if own is None or ref is None:
        missing = ARMS[arm]["name"] if own is None else ref_name
        print(f"{title}: нет прогона {missing!r}\n")
        return
    cmp = aic_stats.compare(own.load_eval(), ref.load_eval(),
                            op=ref.operating_point(), train_sigma=0.0035)
    print(f"--- {title}")
    print("\n".join(cmp.report(ref_name)))
    print()


for arm in ARMS_TO_RUN:
    if arm == "A":
        continue
    verdict_against(arm, ANCHOR, f"[{arm}] против якоря A: даёт ли ветка хоть что-нибудь")
    if arm in COUNTERPART:
        verdict_against(arm, COUNTERPART[arm],
                        f"[{arm}] против пары {COUNTERPART[arm]}")

## 10. Где именно изменилось: по доменам

In [ ]:
# Предсказание серии сформулировано по доменам, поэтому сравнение тоже по ним и
# в ОДНОЙ операционной точке: у каждого прогона своя тюненая точка, и разница
# «в своих точках» включала бы разницу подбора порогов, а не признака.
def domain_table(name: str, at):
    run_dir = ws.runs / name
    return (aic.OofView.from_run(run_dir).by_column("subsource", at)
            if run_dir.exists() else None)


pairs = [(a, COUNTERPART[a]) for a in ARMS_TO_RUN if a in COUNTERPART]
if not pairs:
    print("пары N/P в этом запуске нет — сравнивать по доменам нечего")
for arm, ref_name in pairs:
    ref = open_run(ref_name)
    if ref is None:
        print(f"нет прогона {ref_name!r}")
        continue
    shared_op = ref.operating_point()          # общая точка — точка пары
    mine = domain_table(ARMS[arm]["name"], shared_op)
    theirs = domain_table(ref_name, shared_op)
    if mine is None or theirs is None:
        continue
    delta = pd.DataFrame({
        "n_pos": mine["n_pos"],
        f"dice {arm}": mine["dice_pos"].astype(float).round(4),
        "dice P": theirs["dice_pos"].astype(float).round(4),
        "разница": (mine["dice_pos"].astype(float)
                    - theirs["dice_pos"].astype(float)).round(4),
    })
    print(f"Dice по доменам, {arm} против {ref_name}, общая рабочая точка\n")
    print(delta.sort_values("разница", ascending=False).to_string())
    print("\nОжидание серии: плюс на vision и coco, около нуля на plain_l9.\n")

In [ ]:
last = open_run(ARMS[ARMS_TO_RUN[-1]]["name"])
if last is not None:
    view = aic.OofView.from_run(last.dir)
    op = last.operating_point()
    print(view.by_area(op), "\n")
    print(view.by_column("subsource", op), "\n")
    print(view.ceilings(op))

## 11. Лидерборд прогонов

In [ ]:
print(aic.leaderboard(ws.runs))